<a href="https://colab.research.google.com/github/busycaesar/Attention_Attention_Everywhere/blob/Master/QKV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [2]:
from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2Model.from_pretrained('gpt2', output_attentions=True)
model.eval()

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [3]:
sentence = 'The dog did not cross the road because it was'

In [5]:
inputs = tokenizer(sentence, return_tensors='pt')

tokens = [tokenizer.decode([tid]) for tid in inputs['input_ids'][0]]

it_idx = tokens.index(' it')
dog_idx = tokens.index(' dog')
road_idx = tokens.index(' road')

## Tokens become vectors​

In [6]:
import torch

with torch.no_grad():
    outputs = model(**inputs)

for word, idx in [('dog', dog_idx), ('road', road_idx), ('it', it_idx)]:
    vec = outputs.last_hidden_state[0, idx][:5]
    values = ', '.join(f'{v:+.3f}' for v in vec.tolist())
    print(f'{word} = [{values}, ...]')

dog = [+0.046, -0.308, +0.103, +0.268, -0.141, ...]
road = [+0.230, -0.264, -2.645, -0.136, -0.077, ...]
it = [-0.480, -0.430, -0.443, -0.565, -0.060, ...]


## From output vector to next token prediction​

In [8]:
import torch

with torch.no_grad():
    h = model(**inputs, output_hidden_states=True).hidden_states[-1][0, -1]

print(f'Output vector: [{", ".join(f"{x:+.3f}" for x in h[:8].tolist())}, ...]   ({len(h)} numbers)\n')

top_n_tokens = 100

logits = h @ model.wte.weight.T
vocab_size = model.config.vocab_size
probs  = torch.softmax(logits, dim=-1)
top    = torch.topk(probs, top_n_tokens)

print(f'Scored against all {vocab_size} tokens. top {top_n_tokens}:\n')
for p, i in zip(top.values, top.indices):
    print(f'  {tokenizer.decode([i]):<14} {p*100:5.5f}%')

Output vector: [-0.034, +0.005, -0.586, -0.430, +0.343, -0.048, -1.473, +0.525, ...]   (768 numbers)

Scored against all 50257 tokens. top 100:

   afraid        7.08241%
   not           6.68700%
   too           6.57735%
   scared        5.02134%
   in            2.55453%
   a             2.08128%
   frightened    1.79524%
   running       1.64846%
   so            1.63166%
   trying        1.51279%
   on            1.43290%
   "             1.40881%
   being         1.23015%
   fearful       1.20079%
   worried       1.12296%
   barking       0.87016%
   concerned     0.83490%
   very          0.70275%
   out           0.68660%
   still         0.67822%
   chasing       0.64512%
   unable        0.64239%
   already       0.60128%
   going         0.59668%
   under         0.59285%
   stuck         0.59253%
   injured       0.54569%
   just          0.50219%
   caught        0.46798%
   terrified     0.45203%
   moving        0.44456%
   at            0.42157%
   safe          0.4156

## QK: does the ask match the offer?​

In [ ]:
import torch

qkv_store = {}

def make_hook(layer_idx):
    def hook(module, input, output):
        with torch.no_grad():
            x   = input[0]
            qkv = module.c_attn(x)
            d   = model.config.n_embd
            Q, K, V = qkv.split(d, dim=2)
            qkv_store[layer_idx] = (
                Q.squeeze(0).detach(),
                K.squeeze(0).detach(),
                V.squeeze(0).detach()
            )
    return hook

blocks = model.transformer.h if hasattr(model, 'transformer') else model.h

handles = [blocks[L].attn.register_forward_hook(make_hook(L)) for L in range(len(blocks))]

with torch.no_grad():
    outputs = model(**inputs)

for h in handles:
    h.remove()

In [ ]:
import torch, math

L, H   = 4, 3
d_head = model.config.n_embd // model.config.n_head
Q, K, _ = qkv_store[L]
s, e   = H*d_head, (H+1)*d_head

q_it   = Q[it_idx,   s:e]
k_dog  = K[dog_idx,  s:e]
k_road = K[road_idx, s:e]

N = 5

def row(name, v):
    print(f'  {name:<9} = [{", ".join(f"{x:+.3f}" for x in v[:N].tolist())}, ... ]   ({len(v)} numbers)')

print(f'GPT-2  layer {L}, head {H}\n')
row('q_it',   q_it)
row('k_dog',  k_dog)
row('k_road', k_road)

for nm, k in [('dog', k_dog), ('road', k_road)]:
    print(f'\n  q_it · k_{nm}')
    print('    ' + ' + '.join(f'({q_it[i]:+.3f})({k[i]:+.3f})' for i in range(N)) + ' + ...')
    dot = torch.dot(q_it, k).item()
    print(f'    = {dot:+.4f}      (all {len(q_it)} terms)')

## The full trace​

In [ ]:
print('GPT-2 loaded.')
print(f'Layers : {model.config.n_layer}')
print(f'Heads  : {model.config.n_head}')
print(f'Embedding dimension: {model.config.n_embd}')
print(f'Q, K, V per head: {model.config.n_embd // model.config.n_head}  (d_model / n_heads)')

In [ ]:
def fmt(v, n=4):
    return '[' + ', '.join(f'{x:+.3f}' for x in v[:n].tolist()) + ', ...]'

def fmt_ends(v, n=3):
    h = ', '.join(f'{x:+.3f}' for x in v[:n].tolist())
    t = ', '.join(f'{x:+.3f}' for x in v[-n:].tolist())
    return f'[{h}, ... , {t}]   (len {len(v)})'

with torch.no_grad():
    hs = model(**inputs, output_hidden_states=True).hidden_states

n_layers = model.config.n_layer
n_heads  = model.config.n_head
d_head   = model.config.n_embd // n_heads
prior    = list(range(it_idx + 1))
names    = [tokens[i].strip() for i in prior]

print('='*100)
print('Starting vector for "it"  (vector embedding + positional encoding)')
print('='*100)
print(f'  x_it = {fmt(hs[0][0, it_idx], 10)}   (768 numbers total)')
print('\n  note: each layer also has an MLP sublayer that adds to x_it.')
print('        the tables below trace the attention half only.\n')

for L in range(n_layers):
    print('\n' + '='*100)
    print(f'LAYER {L}')
    print('='*100)

    hdr = f'{"Head":<6}' + ''.join(f'{n:>10}' for n in names)
    print(hdr); print('-'*len(hdr))
    for H in range(n_heads):
        w = outputs.attentions[L][0][H][it_idx][:len(prior)]
        print(f'{H:<6}' + ''.join(f'{v:>10.3f}' for v in w.tolist()))

    _, _, V = qkv_store[L]
    head_outs = []
    for H in range(n_heads):
        w  = outputs.attentions[L][0][H][it_idx]
        Vh = V[:, H*d_head:(H+1)*d_head]
        head_outs.append((w.unsqueeze(1) * Vh).sum(0))

    # ---- show the formula with real numbers for head 0 ----
    w0  = outputs.attentions[L][0][0][it_idx]
    V0  = V[:, 0:d_head]
    print(f'\n  head output = sum of (attention weight x value vector)')
    print(f'  head 0, written out:\n')
    for i in prior:
        print(f'      {w0[i]:+.3f}  x  v_{names[i]:<8} {fmt(V0[i], 3)}')
    print(f'      {"-"*54}')
    print(f'      sum     = {fmt(head_outs[0], 3)}   ({d_head} numbers)')

    z = torch.cat(head_outs)
    print(f'\n  all {n_heads} heads flattened:')
    print(f'      z = {fmt_ends(z)}')

    Wo = blocks[L].attn.c_proj.weight
    bo = blocks[L].attn.c_proj.bias

    print(f'\n  W^O  — one per layer, shared by all {n_heads} heads   (top-left 4x4 of 768x768)')
    for r in range(4):
        print('      ' + ''.join(f'{Wo[r, c]:>9.3f}' for c in range(4)))

    attn_out = z @ Wo + bo
    print(f'\n  z @ W^O  — each output slot is z dotted with one column of W^O')
    terms = ' + '.join(f'({z[k]:+.3f})({Wo[k,0]:+.3f})' for k in range(3))
    print(f'      slot 0 = {terms} + ... + b')
    print(f'             = {attn_out[0]:+.3f}')
    print(f'      attention output = {fmt(attn_out)}')

    x_old = hs[L][0, it_idx]
    print(f'\n  new x_it = old x_it + attention output')
    print(f'      old x_it     {fmt(x_old)}')
    print(f'    + attn output  {fmt(attn_out)}')
    print(f'    ' + '-'*46)
    print(f'    = new x_it     {fmt(x_old + attn_out)}')

In [ ]:
import numpy as np, matplotlib.pyplot as plt

n_layers = model.config.n_layer
n_heads  = model.config.n_head
prior    = list(range(it_idx + 1))
names    = [tokens[i].strip() for i in prior]

fig, axes = plt.subplots(6, 2, figsize=(16, 30))
axes = axes.flatten()

for L in range(n_layers):
    ax   = axes[L]
    grid = np.zeros((n_heads, len(prior)))
    for H in range(n_heads):
        grid[H] = outputs.attentions[L][0][H][it_idx][:len(prior)].detach().numpy()

    im = ax.imshow(grid, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(len(prior)))
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=10)
    ax.set_yticks(range(n_heads))
    ax.set_yticklabels(range(n_heads), fontsize=9)
    ax.set_title(f'Layer {L}', fontsize=12, pad=8)
    if L % 2 == 0:
        ax.set_ylabel('Head', fontsize=10)

    for col, c in [(dog_idx, 'green'), (road_idx, 'red')]:
        ax.axvline(col, color=c, lw=1.3, ls='--', alpha=0.55)

fig.suptitle('Attention FROM "it" TO every previous token — all 12 layers x 12 heads',
             fontsize=15, y=0.999)
fig.colorbar(im, ax=axes.tolist(), label='attention weight', shrink=0.4)
plt.savefig('all_layers_heatmap.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np, matplotlib.pyplot as plt

LAYER, HEAD = 4, 3

prior = list(range(it_idx + 1))
names = [tokens[i].strip() for i in prior]

w = outputs.attentions[LAYER][0][HEAD][it_idx][:len(prior)].detach().numpy()

fig, ax = plt.subplots(figsize=(14, 2.4))
im = ax.imshow(w[np.newaxis, :], aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

ax.set_xticks(range(len(prior)))
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=13)
ax.set_yticks([0]); ax.set_yticklabels([f'Head {HEAD}'], fontsize=12)
ax.set_title(f'Layer {LAYER}, Head {HEAD} — attention FROM "it"', fontsize=15, pad=14)

for i, v in enumerate(w):
    ax.text(i, 0, f'{v:.3f}', ha='center', va='center',
            fontsize=12, fontweight='bold',
            color='white' if v > 0.5 else 'black')

plt.colorbar(im, ax=ax, label='attention weight', shrink=0.9)
plt.tight_layout()
plt.savefig(f'layer{LAYER}_head{HEAD}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'  dog   = {w[dog_idx]:.4f}')
print(f'  road  = {w[road_idx]:.4f}')
print(f'  ratio = {w[dog_idx]/w[road_idx]:.0f}x')